In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("🚀 MODELO DE PREDICCIÓN - MUNDIAL\n")

# =========================================
# ☣ 2. CARGA DE DATOS
# =========================================
base_path = '/content/drive/MyDrive/Simulaciones_Mundial/Data'

cruces = pd.read_csv(f"{base_path}/cruces_16avos.csv")
datos_historicos = pd.read_csv(f"{base_path}/datos_historicos.csv")
datos_mundial = pd.read_csv(f"{base_path}/datos_mundial.csv")
detalle_simulacion = pd.read_csv(f"{base_path}/detalle_simulacion_torneo.csv")
grupos_mundial = pd.read_csv(f"{base_path}/Grupos_Mundial.csv")
partidos_mundial = pd.read_csv(f"{base_path}/partidos_mundial.csv")
partidos = pd.read_csv(f"{base_path}/partidos.csv")
ranking_fifa = pd.read_csv(f"{base_path}/ranking_fifa.csv")
transfermarkt = pd.read_csv(f"{base_path}/transfermarkt.csv")

print("✅ Datos cargados")

# =========================================
# ☣ 3. LIMPIEZA DE NOMBRES Y MAPEO DE PLACEHOLDERS
# =========================================
def clean_name(name):
    name = str(name).strip()
    replacements = {
        'Estados Unidos': 'EE. UU.',
        'Bosnia y Herzegovina': 'Bosnia-Herzegovina',
        'República de Corea': 'Corea del Sur', 'RI de Irán': 'Irán',
        'Islas de Cabo Verde': 'Cabo Verde',
        'Repùblica de Corea': 'Corea del Sur'
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name

# Mapeo de terceros y segundos
team_map = {
    "Third Place Group C/E/F/H/I": "Suecia",
    "Third Place Group E/H/I/J/K": "Ecuador",
    "Third Place Group A/E/H/I/J": "Bosnia-Herzegovina",
    "Third Place Group E/F/G/I/J": "Paraguay",
    "Group J 2nd Place": "Austria",
    "Group L 2nd Place": "Ghana",
    "Group K 2nd Place": "Portugal",
    "Group L Winner": "Inglaterra"
}

def get_real_team(name):
    name = clean_name(str(name))
    return team_map.get(name, name)

if 'País' in ranking_fifa.columns:
    ranking_fifa = ranking_fifa.rename(columns={'País': 'Equipo'})

for df_source in [cruces, datos_mundial, ranking_fifa, transfermarkt]:
    if 'Equipo' in df_source.columns:
        df_source['Equipo'] = df_source['Equipo'].apply(get_real_team)

if 'Equipo_Local' in partidos.columns:
    partidos['Equipo_Local'] = partidos['Equipo_Local'].apply(get_real_team)
    partidos['Equipo_Visitante'] = partidos['Equipo_Visitante'].apply(get_real_team)
if 'Equipo_Local' in datos_historicos.columns:
    datos_historicos['Equipo_Local'] = datos_historicos['Equipo_Local'].apply(get_real_team)
    datos_historicos['Equipo_Visitante'] = datos_historicos['Equipo_Visitante'].apply(get_real_team)

# =========================================
# ☣ 4. DATASET DE EQUIPOS
# =========================================
def get_numeric(df):
    cols = ['Equipo'] + df.select_dtypes(include=[np.number]).columns.tolist()
    return df[cols]

stats = (get_numeric(ranking_fifa)
    .merge(get_numeric(transfermarkt), on='Equipo', how='left')\
    .merge(get_numeric(datos_mundial), on='Equipo', how='left'))

stats = stats.fillna(0)

print("📊 Stats:", stats.shape)

# =========================================
# ☣ 5. CREAR DATASET DE ENTRENAMIENTO REAL
datos_historicos['home'] = datos_historicos['Equipo_Local'].apply(get_real_team)
datos_historicos['away'] = datos_historicos['Equipo_Visitante'].apply(get_real_team)

data = []

for _, row in datos_historicos.iterrows():
    h = row['home']
    a = row['away']

    s_h = stats[stats['Equipo'] == h]
    s_a = stats[stats['Equipo'] == a]

    if s_h.empty or s_a.empty:
        continue

    s_h = s_h.iloc[0]
    s_a = s_a.iloc[0]

    match = {}

    for col in stats.select_dtypes(include=[np.number]).columns:
        match[f"{col}_diff"] = s_h[col] - s_a[col]

    goles_h = row['Goles_Local']
    goles_a = row['Goles_Visitante']

    if goles_h > goles_a:
        match['target'] = 1
    elif goles_h < goles_a:
        match['target'] = -1
    else:
        match['target'] = 0 # Draw

    data.append(match)

df_train = pd.DataFrame(data)

print("📈 Dataset entrenamiento:", df_train.shape)

# =========================================
# ☣ 6. MODELO
# =========================================
X = df_train.drop(columns=['target'])
y = df_train['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = RandomForestClassifier(n_estimators=500, max_depth=12, random_state=42)
model.fit(X_scaled, y)

print("✅ Modelo entrenado con datos reales")

# =========================================
# ☣ 7. PREPARAR CRUCES
# =========================================
def extract_teams_from_match_string(match_string):
    match_string = str(match_string).strip()
    parts = re.split(r'\s*vs\.?' + '|' + r'\s+-\s+' + '|' + r'\s+\–\s+' + '|' + r'\s+\—\s+', match_string)
    teams = [get_real_team(p.strip()) for p in parts if p.strip()]
    if len(teams) >= 2:
        return teams[0], teams[1]
    return None, None

cruces[['home','away']] = cruces['Partido'].apply(lambda x: pd.Series(extract_teams_from_match_string(x)))

# =========================================
# ☣ 8. PREDICCIÓN + PROBABILIDADES
# =========================================
print("\n🏆 PREDICCIONES\n")
clasificados = []

for _, row in cruces.iterrows():

    home = row['home']
    away = row['away']

    s_h = stats[stats['Equipo'] == home]
    s_a = stats[stats['Equipo'] == away]

    if s_h.empty or s_a.empty:
        print(f"⚠️ Datos faltantes: {home} vs {away}")
        continue

    s_h = s_h.iloc[0]
    s_a = s_a.iloc[0]

    match = {}

    for col in stats.select_dtypes(include=[np.number]).columns:
        match[f"{col}_diff"] = s_h[col] - s_a[col]

    X_pred = pd.DataFrame([match])
    X_pred_scaled = scaler.transform(X_pred)

    probs = model.predict_proba(X_pred_scaled)[0]

    home_win_prob_idx = np.where(model.classes_ == 1)[0][0]
    away_win_prob_idx = np.where(model.classes_ == -1)[0][0]
    draw_prob_idx = np.where(model.classes_ == 0)[0][0]

    p_home = probs[home_win_prob_idx]
    p_away = probs[away_win_prob_idx]
    p_draw = probs[draw_prob_idx]

    # =========================================
    # ☣ POISSON (MARCADOR ESPERADO)
    # =========================================
    lam_home = 1.2 + (p_home * 2)
    lam_away = 1.2 + (p_away * 2)

    goles_home = np.random.poisson(lam_home)
    goles_away = np.random.poisson(lam_away)

    if p_home > p_away and p_home > p_draw:
        ganador = home
    elif p_away > p_home and p_away > p_draw:
        ganador = away
    else:
        ganador = home if goles_home > goles_away else (away if goles_away > goles_home else home) # Default to home on perfect tie

    clasificados.append(ganador)

    print(f"{home:20} {goles_home} - {goles_away} {away}")
    print(f"   Probabilidades → {home}: {p_home:.2%} | Empate: {p_draw:.2%} | {away}: {p_away:.2%}")
    print(f"   ➡️ Clasifica: {ganador}\n")

# =========================================
# ☣ 9. RESULTADO FINAL
# =========================================
print("🎯 CLASIFICADOS")
for t in clasificados:
    print("✅", t)

🚀 MODELO DE PREDICCIÓN - MUNDIAL

✅ Datos cargados
📊 Stats: (6931, 47)
📈 Dataset entrenamiento: (1241, 47)
✅ Modelo entrenado con datos reales

🏆 PREDICCIONES

Sudáfrica            3 - 3 Canadá
   Probabilidades → Sudáfrica: 27.58% | Empate: 34.79% | Canadá: 37.63%
   ➡️ Clasifica: Canadá

Brasil               0 - 2 Japón
   Probabilidades → Brasil: 47.08% | Empate: 26.96% | Japón: 25.96%
   ➡️ Clasifica: Brasil

Alemania             2 - 0 Paraguay
   Probabilidades → Alemania: 68.43% | Empate: 26.06% | Paraguay: 5.51%
   ➡️ Clasifica: Alemania

Países Bajos         1 - 1 Marruecos
   Probabilidades → Países Bajos: 36.45% | Empate: 33.14% | Marruecos: 30.41%
   ➡️ Clasifica: Países Bajos

Costa de Marfil      1 - 2 Noruega
   Probabilidades → Costa de Marfil: 34.18% | Empate: 38.99% | Noruega: 26.83%
   ➡️ Clasifica: Noruega

Francia              5 - 2 Suecia
   Probabilidades → Francia: 68.96% | Empate: 22.92% | Suecia: 8.13%
   ➡️ Clasifica: Francia

México               2 - 0 Suecia